In [ ]:
import matplotlib.pyplot as plt
from rank import rank_qoe_3_classes, rank_str_to_int_mapping_3_classes
import pandas as pd
import matplotlib.pyplot as plt
import ast
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
from rank import rank_qoe, rank_str_to_int_mapping

from rank import rank_qoe_3_classes, rank_str_to_int_mapping_3_classes

In [ ]:
FEATURE_COLUMNS = [
            'src2dst_avg_pkt_size', 'src2dst_avg_pkt_size_mean', 'src2dst_avg_pkt_size_median',
            'src2dst_avg_pkt_size_std', 'src2dst_avg_pkt_size_min', 'src2dst_avg_pkt_size_max',
            'src2dst_pps', 'src2dst_pps_mean', 'src2dst_pps_median',
            'src2dst_pps_std', 'src2dst_pps_min', 'src2dst_pps_max',
            'dst2src_avg_pkt_size', 'dst2src_avg_pkt_size_mean', 'dst2src_avg_pkt_size_median',
            'dst2src_avg_pkt_size_std', 'dst2src_avg_pkt_size_min', 'dst2src_avg_pkt_size_max',
            'dst2src_pps', 'dst2src_pps_mean', 'dst2src_pps_median',
            'dst2src_pps_std', 'dst2src_pps_min', 'dst2src_pps_max',
            'pkt_dir_ratio', 'pkt_dir_ratio_mean', 'pkt_dir_ratio_median',
            'pkt_dir_ratio_std', 'pkt_dir_ratio_min', 'pkt_dir_ratio_max']
FEATURES=FEATURE_COLUMNS+['qoe_score']
files = [
    "output/subset_window_1754898203.csv",
    "output/window_3_1756030271.csv",
    "output/window_5_1756032959.csv","output/window_10_1754904064.csv","output/window_15_1756035248.csv"
]

In [ ]:
df = pd.read_csv(
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\combined_dataset.csv"
)

metadata = pd.read_csv(r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\metadata_results_auto.csv")


In [ ]:


df=df.dropna(subset=['ping'])
df['qoe_score'] = df['ping'].apply(lambda p: rank_str_to_int_mapping_3_classes[rank_qoe_3_classes(p)])
counts = df['qoe_score'].value_counts().sort_index()

print("Counts per class:")
print(counts)
counts = df['qoe_score'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution by Label')
plt.axis('equal')  
plt.show()

df=df.dropna(subset=['ping'])
df['qoe_score'] = df['ping'].apply(lambda p: rank_str_to_int_mapping[rank_qoe(p)])

counts = df['qoe_score'].value_counts()

plt.figure(figsize=(6,6))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution by Label')
plt.axis('equal')  
plt.show()

counts = df['qoe_score'].value_counts().sort_index()

print("Counts per class:")
print(counts)

In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt

from rank import rank_qoe_3_classes, rank_str_to_int_mapping_3_classes

def hhmmss_to_seconds(hhmmss: int) -> int:
    h = hhmmss // 10000
    m = (hhmmss // 100) % 100
    s = hhmmss % 100
    return h * 3600 + m * 60 + s
def to_qoe_class3_zero(ping):
    # מחזיר 0/1/2/3 (0 = חסר/לא תקין)
    if pd.isna(ping):
        return 0
    try:
        ping = float(ping)
    except Exception:
        return 0

    label = rank_qoe_3_classes(ping)
    return rank_str_to_int_mapping_3_classes.get(label, 0)  # אם "Error" -> 0

def to_qoe_class5_zero(ping):
    # מחזיר 0/1/2/3 (0 = חסר/לא תקין)
    if pd.isna(ping):
        return 0
    try:
        ping = float(ping)
    except Exception:
        return 0
    label = rank_qoe(ping)
    return rank_str_to_int_mapping.get(label, 0)  # אם "Error" -> 0


In [ ]:


def build_series_from_row3(row):
    folder = row["folder"]
    start_hhmmss = int(folder.split("_")[-1])
    start_sec = hhmmss_to_seconds(start_hhmmss)

    pairs = ast.literal_eval(row["time_value_pairs"])

    xs, ys = [], []
    for t_str, ping in pairs.items():
        t = int(t_str)
        dt = hhmmss_to_seconds(t) - start_sec
        if dt < 0:
            continue
        # if(to_qoe_class_zero(ping)!=0):
        xs.append(dt)
        ys.append(to_qoe_class3_zero(ping))

    return pd.DataFrame({"t_sec": xs, "qoe": ys}).sort_values("t_sec")


In [ ]:

row0 = metadata.iloc[40]
s0 = build_series_from_row3(row0)
mask0 = (s0["qoe"] == 0)

plt.figure(figsize=(12,4))

# קו כחול אבל בלי לחבר דרך ה-0 (NaN שוברים את הקו)
qoe_line = s0["qoe"].where(~mask0, np.nan)
plt.plot(s0["t_sec"], qoe_line, marker="o", linestyle="-", label="qoe (1/2/3)")

# נקודות אדומות ל-0 (בלי קו)
plt.scatter(s0.loc[mask0, "t_sec"], s0.loc[mask0, "qoe"],
            color="red", zorder=5, label="qoe=0(null)")

plt.yticks([0,1,2,3])
plt.ylim(-0.5, 3.5)
plt.xlabel("Time (seconds from run start)")
plt.ylabel("QoE class (0/1/2/3)")
plt.title(f"QoE over time - {row0['folder']} / {row0['pcapng_file']}")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:

def build_series_from_row5(row):
    folder = row["folder"]
    start_hhmmss = int(folder.split("_")[-1])
    start_sec = hhmmss_to_seconds(start_hhmmss)

    pairs = ast.literal_eval(row["time_value_pairs"])

    xs, ys = [], []
    for t_str, ping in pairs.items():
        t = int(t_str)
        dt = hhmmss_to_seconds(t) - start_sec
        if dt < 0:
            continue
        # if(to_qoe_class_zero(ping)!=0):
        xs.append(dt)
        ys.append(to_qoe_class5_zero(ping))

    return pd.DataFrame({"t_sec": xs, "qoe": ys}).sort_values("t_sec")


In [ ]:
import numpy as np

row0 = metadata.iloc[120]
s0 = build_series_from_row5(row0)
mask0 = (s0["qoe"] == 0)

plt.figure(figsize=(12,4))

# קו כחול אבל בלי לחבר דרך ה-0 (NaN שוברים את הקו)
qoe_line = s0["qoe"].where(~mask0, np.nan)
plt.plot(s0["t_sec"], qoe_line, marker="o", linestyle="-", label="qoe (1/2/3/4/5)")

# נקודות אדומות ל-0 (בלי קו)
plt.scatter(s0.loc[mask0, "t_sec"], s0.loc[mask0, "qoe"],
            color="red", zorder=5, label="qoe=0(null)")

plt.yticks([0,1,2,3,4,5])
plt.ylim(-0.5, 5.5)
plt.xlabel("Time (seconds from run start)")
plt.ylabel("QoE class (0/1/2/3/4/5)")
plt.title(f"QoE over time - {row0['folder']} / {row0['pcapng_file']}")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:


def compute_transitions(series_df):
    """
    מקבל dataframe עם עמודת qoe
    מחזיר Counter של כל המעברים (from, to)
    """
    q = series_df["qoe"].values
    transitions = []

    for i in range(len(q) - 1):
        a, b = q[i], q[i+1]
        if a == 0 or b == 0:
            continue  # מדלגים על null
        transitions.append((a, b))

    return Counter(transitions)
global_counter = Counter()

for _, row in metadata.iterrows():
    s = build_series_from_row3(row)
    c = compute_transitions(s)
    global_counter.update(c)
classes = [1,2,3]

matrix = pd.DataFrame(0, index=classes, columns=classes)

for (a,b), count in global_counter.items():
    if a in classes and b in classes:
        matrix.loc[a,b] += count
matrix_percent = matrix.div(matrix.sum(axis=1), axis=0) * 100
print("Transition Counts:")
print(matrix)

print("\nTransition Percentages (%):")
print(matrix_percent.round(1))
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
sns.heatmap(matrix_percent, annot=True, fmt=".1f", cmap="Blues")
plt.xlabel("To class")
plt.ylabel("From class")
plt.title("QoE Transition Matrix (%)")
plt.show()


In [ ]:
from collections import Counter
import numpy as np

def compute_transitions(series_df):
    """
    מקבל dataframe עם עמודת qoe
    מחזיר Counter של כל המעברים (from, to)
    """
    q = series_df["qoe"].values
    transitions = []

    for i in range(len(q) - 1):
        a, b = q[i], q[i+1]
        if a == 0 or b == 0:
            continue  # מדלגים על null
        transitions.append((a, b))

    return Counter(transitions)
global_counter = Counter()

for _, row in metadata.iterrows():
    s = build_series_from_row5(row)
    c = compute_transitions(s)
    global_counter.update(c)
classes = [1,2,3,4,5]

matrix = pd.DataFrame(0, index=classes, columns=classes)

for (a,b), count in global_counter.items():
    if a in classes and b in classes:
        matrix.loc[a,b] += count
matrix_percent = matrix.div(matrix.sum(axis=1), axis=0) * 100
print("Transition Counts:")
print(matrix)

print("\nTransition Percentages (%):")
print(matrix_percent.round(1))
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
sns.heatmap(matrix_percent, annot=True, fmt=".1f", cmap="Blues")
plt.xlabel("To class")
plt.ylabel("From class")
plt.title("QoE Transition Matrix (%)")
plt.show()


In [ ]:

def to_qoe_class_zero(ping):
    # מחזיר 0/1/2/3 (0 = חסר/לא תקין)
    if pd.isna(ping):
        return 0
    try:
        ping = float(ping)
    except Exception:
        return 0

    label = rank_qoe_3_classes(ping)
    return rank_str_to_int_mapping_3_classes.get(label, 0)  # אם "Error" -> 0
def sume_ech_class(row):
    folder = row["folder"]
    pairs = ast.literal_eval(row["time_value_pairs"])

    class_1=[]
    class_2=[]
    class_3=[]
    for _t_str, ping in pairs.items():
        if ping== None:
            continue
        else:
            q = to_qoe_class_zero(ping)

            if q == 1:
                class_1.append(ping)
            elif q == 2:
                class_2.append(ping)
            elif q == 3:
                class_3.append(ping)


    return class_1, class_2, class_3

all_c1, all_c2, all_c3 = [], [], []

for i in metadata.index:
    row = metadata.loc[i]
    c1, c2, c3 = sume_ech_class(row)
    all_c1.append(len(c1))
    all_c2.append(len(c2))
    all_c3.append(len(c3))
print(f"Class 1 count: {sum(all_c1)}, Class 2 count: {sum(all_c2)}, Class 3 count: {sum(all_c3)}")
print(f"Class 1 std: {np.std(all_c1)}, Class 2 std: {np.std(all_c2)}, Class 3 std: {np.std(all_c3)}")






In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# corr = df[FEATURE_COLUMNS].corr(method='spearman')
corr = df[FEATURES].corr(method='spearman')

plt.figure(figsize=(14, 12))
sns.heatmap(
    corr,
    cmap='coolwarm',
    center=0,
    square=True,
    cbar_kws={'shrink': 0.8}
)
plt.title("Spearman Correlation Between Features")
plt.show()



קורולציה בין פיצרים כללי

In [ ]:

# plt.scatter(df['rtt'], df['packet_loss'])
# plt.xlabel('RTT')
# plt.ylabel('Packet Loss')
import numpy as np
import pandas as pd


# from itertools import combinations
# for a, b in combinations(FEATURE_COLUMNS, 2):
#     print(a, b)
#     plt.scatter(df[a], df[b])
#     plt.xlabel(a)
#     plt.ylabel(b)
#     plt.show()
#     df[[a,  b]].corr(method='pearson')



corr_features = df[FEATURE_COLUMNS+['qoe_score']].corr(method='spearman')

threshold = 0.85
high_corr = []

for i in range(len(corr_features.columns)):
    for j in range(i + 1, len(corr_features.columns)):
        val = corr_features.iloc[i, j]
        if abs(val) > threshold:
            high_corr.append((
                corr_features.columns[i],
                corr_features.columns[j],
                val
            ))

high_corr_df = pd.DataFrame(high_corr, columns=['feature_1', 'feature_2', 'spearman_corr'])
high_corr_df = high_corr_df.sort_values(by='spearman_corr', key=np.abs, ascending=False)
print(high_corr_df.head(20))




In [ ]:
# for _, row in high_corr_df.iterrows():
#     a = row['feature_1']
#     plt.scatter(df[a], df["qoe_score"], alpha=0.3)
#     plt.xlabel(a)
#     plt.ylabel("qoe_score")
#     plt.title(f"Spearman={row['spearman_corr']:.2f}")
#     plt.show()

In [ ]:
x ="dst2src_avg_pkt_size_median"
plt.scatter(df[x], df["qoe_score"], alpha=0.3)
plt.xlabel(x)
plt.ylabel("qoe_score")
plt.title(f"Spearman={row['spearman_corr']:.2f}")
plt.show()

In [ ]:
for _, row in high_corr_df.head(5).iterrows():
    a, b = row['feature_1'], row['feature_2']
    plt.scatter(df[a], df[b], alpha=0.3)
    plt.xlabel(a)
    plt.ylabel(b)
    plt.title(f"Spearman={row['spearman_corr']:.2f}")
    plt.show()


קורולציה בין פיצרים פר מחלקה


In [ ]:
print("\n=== Correlation by classes ===")
from rank import rank_qoe_3_classes, rank_str_to_int_mapping_3_classes
df = df.dropna(subset=['ping'])
df['qoe_score'] = df['ping'].apply(lambda p: rank_str_to_int_mapping_3_classes[rank_qoe_3_classes(p)])

for label in sorted(df['qoe_score'].unique()):
    sub = df[df['qoe_score'] == label]
    corr = sub[FEATURE_COLUMNS].corr(method='spearman')
    print(f"\n=== Class {label} ===")
    print(corr.mean().sort_values(ascending=False).head(5))


מפת חום

קורולציה של המחלקות 

In [ ]:
df.corr(method='spearman')['qoe_score'].sort_values(ascending=False)


אילו פיצרים באמת קשורים

In [ ]:
per_class_top = {}

for cls in sorted(df['qoe_score'].unique()):
    sub = df[df['qoe_score'] == cls]
    corr = sub[FEATURE_COLUMNS].corr(method='spearman')
    per_class_top[cls] = corr.abs().mean()

df_compare = pd.DataFrame(per_class_top)
df_compare['range'] = df_compare.max(axis=1) - df_compare.min(axis=1)

df_compare.sort_values('range', ascending=False).head(10)
#זה מראה שדפוסי ההתנהגות של הרשת משתנים בין איכות חוויה טובה, בינונית וגרועה.

In [ ]:
import numpy as np
for col in FEATURES:
    plt.figure()

    data_all = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    plt.hist(data_all, bins=40, alpha=0.3, label="All QoE", density=True)

    for cls in sorted(df['qoe_score'].unique()):
        data = df.loc[df['qoe_score'] == cls, col].replace([np.inf, -np.inf], np.nan).dropna()
        plt.hist(data, bins=40, alpha=0.5, label=f"QoE {cls}", density=True)

    plt.title(col)
    plt.legend()
    plt.show()


In [ ]:

for col in FEATURE_COLUMNS:
    plt.figure()
    for cls in sorted(df['qoe_score'].unique()):
        sns.kdeplot(
            df[df['qoe_score'] == cls][col].dropna(),
            label=f"QoE {cls}",
            fill=True
        )
    plt.title(col)
    plt.legend()
    plt.show()


In [ ]:
# from modelse import make_models
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.inspection import permutation_importance

# def top_features_per_model(
#     df: pd.DataFrame,
#     models_list,                 # list of (name, model)
#     feature_cols,
#     target_col: str,
#     test_size: float = 0.2,
#     random_state: int = 42,
#     n_repeats: int = 10,
#     top_k: int = 10
# ):
#     # Data
#     X = df[feature_cols].astype("float32")
#     y = df[target_col].astype(int)

#     # stratify only if each class appears enough times
#     strat = y if y.nunique() > 1 else None

#     X_train, X_test, y_train, y_test = train_test_split(
#         X, y,
#         test_size=test_size,
#         random_state=random_state,
#         stratify=strat
#     )

#     rows = []

#     for name, model in models_list:
#         try:
#             model.fit(X_train, y_train)

#             perm = permutation_importance(
#                 model,
#                 X_test,
#                 y_test,
#                 n_repeats=n_repeats,
#                 random_state=random_state,
#                 n_jobs=-1
#             )

#             importances = pd.Series(perm.importances_mean, index=feature_cols)
#             top = importances.sort_values(ascending=False).head(top_k)

#             for feat, val in top.items():
#                 rows.append({
#                     "model": name,
#                     "feature": feat,
#                     "perm_importance_mean": float(val)
#                 })

#         except Exception as e:
#             # אם מודל נופל – לא עוצרים הכל
#             rows.append({
#                 "model": name,
#                 "feature": "__ERROR__",
#                 "perm_importance_mean": np.nan
#             })
#             print(f"[WARN] {name} failed: {e}")

#     out = pd.DataFrame(rows)

#     # תצוגה יפה: כל מודל עם ה-top שלו
#     return out.sort_values(["model", "perm_importance_mean"], ascending=[True, False])
# # אחרי שאתה יוצר df['y'] שלך שהוא 1/2/3:
# df['qoe_score'] = df['qoe_score'] - 1

# models_list = make_models(n_classes=3, knn_k=3, knn_weights='uniform', knn_metric='minkowski')

# imp_df = top_features_per_model(
#     df=df,
#     models_list=models_list,
#     feature_cols=FEATURE_COLUMNS,
#     target_col="qoe_score",         
#     n_repeats=10,
#     top_k=10
# )

# print(imp_df)


In [ ]:
import os, ast, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scapy.all import PcapNgReader, TCP, UDP


# ---------- 1) Parsing helpers ----------

def extract_run_date(folder: str):
    """Extract YYYYMMDD from folder name (anywhere). Return date or None."""
    m = re.search(r"(\d{8})", folder)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1), "%Y%m%d").date()
    except ValueError:
        return None


def get_relevant_window_ts(folder: str, time_value_pairs: dict):
    """
    Return (start_ts, end_ts) for the relevant window:
    first non-None HHMMSS to last non-None HHMMSS, using date from folder.
    """
    good_times = [t for t, v in time_value_pairs.items() if v is not None]
    if not good_times:
        return None

    run_date = extract_run_date(folder)
    if run_date is None:
        return None

    start_hhmmss = str(good_times[0]).zfill(6)
    end_hhmmss   = str(good_times[-1]).zfill(6)

    start_dt = datetime.combine(run_date, datetime.strptime(start_hhmmss, "%H%M%S").time())
    end_dt   = datetime.combine(run_date, datetime.strptime(end_hhmmss, "%H%M%S").time())

    return start_dt.timestamp(), end_dt.timestamp()


# ---------- 2) Core runner: iterate packets in window and aggregate ----------

def run_metric_for_group(metadata_csv: str, pcaps_root: str, metric_fn):
    """
    metric_fn(reader, start_ts, end_ts) -> list/tuple/number, depends on metric.
    We return a list of metric outputs per capture, AND a merged list (if metric returns lists).
    """
    df = pd.read_csv(metadata_csv)

    per_capture = []   # store metric output per capture (could be list or tuple)
    merged = []        # if metric returns list, we extend into merged
    used = 0

    for _, row in df.iterrows():
        folder = str(row["folder"])
        pcap_file = str(row["pcapng_file"])

        try:
            pairs = ast.literal_eval(row["time_value_pairs"])
            if not isinstance(pairs, dict):
                continue
        except Exception:
            continue

        window = get_relevant_window_ts(folder, pairs)
        if window is None:
            continue

        start_ts, end_ts = window
        pcap_path = os.path.join(pcaps_root, folder, pcap_file)
        if not os.path.exists(pcap_path):
            continue

        with PcapNgReader(pcap_path) as reader:
            out = metric_fn(reader, start_ts, end_ts)

        if out is None:
            continue

        per_capture.append(out)
        used += 1

        # אם זה list (IAT / packet sizes) - נצבור גם merged
        if isinstance(out, list):
            merged.extend(out)

    return used, per_capture, merged


# ---------- 3) Metrics (choose what you want) ----------

def metric_iat(reader, start_ts, end_ts, max_iat_sec=1.0):
    """Return list of inter-arrival times (sec) inside window."""
    iats = []
    last_t = None
    for pkt in reader:
        t = float(pkt.time)
        if t < start_ts:
            continue
        if t > end_ts:
            break

        if last_t is not None:
            dt = t - last_t
            if dt >= 0 and (max_iat_sec is None or dt <= max_iat_sec):
                iats.append(dt)
        last_t = t
    return iats


def metric_pkt_sizes(reader, start_ts, end_ts):
    """Return list of packet sizes (bytes) inside window."""
    sizes = []
    for pkt in reader:
        t = float(pkt.time)
        if t < start_ts:
            continue
        if t > end_ts:
            break
        sizes.append(len(pkt))
    return sizes


def metric_tcp_udp_counts(reader, start_ts, end_ts):
    """Return tuple (tcp_count, udp_count) inside window."""
    tcp = udp = 0
    for pkt in reader:
        t = float(pkt.time)
        if t < start_ts:
            continue
        if t > end_ts:
            break
        if pkt.haslayer(TCP):
            tcp += 1
        elif pkt.haslayer(UDP):
            udp += 1
    return (tcp, udp)


# ---------- 4) Plotting helpers ----------

def plot_hist_per_group(group_to_values, titles, bins, xlabel, title_prefix, logx=False, density=False):
    """Plot one histogram per group + one overlay plot."""
    # per-group
    for g in titles:
        vals = group_to_values.get(g, [])
        if not vals:
            print(f"{g}: no samples -> skip plot")
            continue
        plt.figure(figsize=(9, 5))
        plt.hist(vals, bins=bins, density=density)
        if logx:
            plt.xscale("log")
        plt.xlabel(xlabel)
        plt.ylabel("Density" if density else "Count")
        plt.title(f"{title_prefix} - {g}")
        plt.show()

    # overlay compare
    plt.figure(figsize=(10, 5))
    any_data = False
    for g in titles:
        vals = group_to_values.get(g, [])
        if not vals:
            continue
        any_data = True
        plt.hist(vals, bins=bins, histtype="step", linewidth=2, label=g, density=True)
    if any_data:
        if logx:
            plt.xscale("log")
        plt.xlabel(xlabel)
        plt.ylabel("Density")
        plt.title(f"{title_prefix} - Comparison")
        plt.legend()
        plt.show()


# ---------- 5) Example: run IAT + PacketSize for multiple groups ----------

metadata_csvs = [
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+0_metadata_results.csv",
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\metadata_results_P+0+100.csv",
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+200_metadata_results.csv",
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+300_metadata_results.csv",
]

pcaps_roots = [
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+0_lbld",
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+100_(50+50)_lbld",
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+200_(100+100)_lbld",
    r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+300_(150+150)_lbld",
]

titles = ["P+0", "P+100", "P+200", "P+300"]

# ----- IAT -----
iat_by_group = {}
for meta, root, t in zip(metadata_csvs, pcaps_roots, titles):
    used, per_cap, merged = run_metric_for_group(
        meta, root,
        metric_fn=lambda reader, s, e: metric_iat(reader, s, e, max_iat_sec=1.0)
    )
    print(f"{t} IAT: captures_used={used}, samples={len(merged)}")
    iat_by_group[t] = merged

plot_hist_per_group(
    iat_by_group, titles,
    bins=120,
    xlabel="Inter-Arrival Time (sec)",
    title_prefix="IAT Distribution (relevant windows only)",
    logx=True
)

# ----- Packet Size -----
size_by_group = {}
for meta, root, t in zip(metadata_csvs, pcaps_roots, titles):
    used, per_cap, merged = run_metric_for_group(
        meta, root,
        metric_fn=metric_pkt_sizes
    )
    print(f"{t} Sizes: captures_used={used}, samples={len(merged)}")
    size_by_group[t] = merged

plot_hist_per_group(
    size_by_group, titles,
    bins=120,
    xlabel="Packet Size (bytes)",
    title_prefix="Packet Size Distribution (relevant windows only)",
    logx=False
)

TCP UDP

הבדלים בין מחלקות ומשחקים 

In [ ]:

for metadata_csv, pcaps_root, title in zip(metadata_csvs, pcaps_roots, titles):
    df = pd.read_csv(metadata_csv)

    tcp_sum = udp_sum = total_sum = 0
    used = 0

    for _, row in df.iterrows():
        folder = str(row["folder"])
        pcap_file = str(row["pcapng_file"])

        try:
            pairs = ast.literal_eval(row["time_value_pairs"])
        except Exception:
            continue

        good_times = [t for t, v in pairs.items() if v is not None]
        if not good_times:
            continue

        start_hhmmss = str(good_times[0]).zfill(6)
        end_hhmmss   = str(good_times[-1]).zfill(6)

        # חילוץ תאריך בצורה בטוחה
        m = re.search(r"(\d{8})", folder)
        if not m:
            continue
        yyyymmdd = m.group(1)

        try:
            run_date = datetime.strptime(yyyymmdd, "%Y%m%d").date()
        except ValueError:
            continue

        start_dt = datetime.combine(run_date, datetime.strptime(start_hhmmss, "%H%M%S").time())
        end_dt   = datetime.combine(run_date, datetime.strptime(end_hhmmss, "%H%M%S").time())

        start_ts = start_dt.timestamp()
        end_ts   = end_dt.timestamp()

        pcap_path = os.path.join(pcaps_root, folder, pcap_file)
        if not os.path.exists(pcap_path):
            # print("Missing:", pcap_path)
            continue

        tcp = udp = total = 0
        with PcapNgReader(pcap_path) as reader:
            for pkt in reader:
                t = float(pkt.time)
                if t < start_ts:
                    continue
                if t > end_ts:
                    break

                total += 1
                if pkt.haslayer(TCP):
                    tcp += 1
                elif pkt.haslayer(UDP):
                    udp += 1

        tcp_sum += tcp
        udp_sum += udp
        total_sum += total
        used += 1

    print(f"{title}: files_used = {used} | total_packets = {tcp_sum + udp_sum}")

    total_pkts = tcp_sum + udp_sum
    if total_pkts == 0:
        print(f"{title}: no TCP/UDP packets in selected windows -> skipping plot")
        continue

    plt.figure(figsize=(5,5))
    plt.pie([tcp_sum, udp_sum], labels=['TCP', 'UDP'], autopct='%1.1f%%')
    plt.title(f"TCP vs UDP Distribution ({title})")
    plt.show()

Inter Arrival TIME Distribution

In [ ]:

# כמה bins להיסטוגרמה
BINS = 120

# כדי להתמקד בזמני IAT "רלוונטים" (למשל עד 1 שניה). אם אתה רוצה הכל, שים None
MAX_IAT_SEC = 1.0



# ====== עזר: IAT בתוך חלון זמן ======
def iats_in_time_window(pcap_path: str, start_ts: float, end_ts: float):
    iats = []
    last_t = None

    with PcapNgReader(pcap_path) as reader:
        for pkt in reader:
            t = float(pkt.time)

            if t < start_ts:
                continue
            if t > end_ts:
                break

            if last_t is not None:
                dt = t - last_t
                if dt >= 0:
                    if (MAX_IAT_SEC is None) or (dt <= MAX_IAT_SEC):
                        iats.append(dt)
            last_t = t

    return iats

# ====== ריצה: לכל קבוצה ======
all_group_iats = {}  # title -> list of iats

for metadata_csv, pcaps_root, title in zip(metadata_csvs, pcaps_roots, titles):
    df = pd.read_csv(metadata_csv)
    group_iats = []
    used_files = 0

    for _, row in df.iterrows():
        folder = str(row["folder"])
        pcap_file = str(row["pcapng_file"])

        # parse time_value_pairs
        try:
            pairs = ast.literal_eval(row["time_value_pairs"])
            if not isinstance(pairs, dict):
                continue
        except Exception:
            continue

        # קח רק timestamps שהערך לא None
        good_times = [t for t, v in pairs.items() if v is not None]
        if not good_times:
            continue

        start_hhmmss = str(good_times[0]).zfill(6)
        end_hhmmss   = str(good_times[-1]).zfill(6)

        run_date = extract_run_date(folder)
        if run_date is None:
            # אם אין תאריך בתיקייה – אי אפשר לבנות epoch מדויק מה-HHMMSS
            # במקרה כזה עדיף לעבוד באופסטים מהתחלת הקובץ (כמו pktIdx אצלך ב-analyze.py).
            # כאן פשוט מדלגים כדי לא להכניס טעות.
            continue

        start_dt = datetime.combine(run_date, datetime.strptime(start_hhmmss, "%H%M%S").time())
        end_dt   = datetime.combine(run_date, datetime.strptime(end_hhmmss, "%H%M%S").time())
        start_ts = start_dt.timestamp()
        end_ts   = end_dt.timestamp()

        pcap_path = os.path.join(pcaps_root, folder, pcap_file)
        if not os.path.exists(pcap_path):
            continue

        iats = iats_in_time_window(pcap_path, start_ts, end_ts)
        if iats:
            group_iats.extend(iats)
            used_files += 1

    all_group_iats[title] = group_iats
    print(f"{title}: captures_used={used_files}, iat_samples={len(group_iats)}")

    # ----- גרף לכל קבוצה -----
    if len(group_iats) == 0:
        print(f"{title}: no IAT samples -> skip plot")
        continue

    plt.figure(figsize=(9, 5))
    plt.hist(group_iats, bins=BINS)
    plt.xscale("log")  # מאוד עוזר ל-IAT
    plt.xlabel("Inter-Arrival Time (sec, log scale)")
    plt.ylabel("Count")
    mx = "" if MAX_IAT_SEC is None else f" (IAT<= {MAX_IAT_SEC}s)"
    plt.title(f"IAT Distribution - {title}{mx}")
    plt.show()

# ====== אופציונלי: גרף השוואה על אותו ציור (overlay) ======
plt.figure(figsize=(10, 5))
for title in titles:
    iats = all_group_iats.get(title, [])
    if not iats:
        continue
    plt.hist(iats, bins=BINS, histtype="step", linewidth=2, label=title, density=True)

plt.xscale("log")
plt.xlabel("Inter-Arrival Time (sec, log scale)")
plt.ylabel("Density")
mx = "" if MAX_IAT_SEC is None else f" (IAT<= {MAX_IAT_SEC}s)"
plt.title(f"IAT Distribution Comparison{mx}")
plt.legend()
plt.show()

Packet Size Distribution

In [ ]:

BINS = 120

def packet_sizes_window(pcap_path, start_ts, end_ts):

    sizes = []

    with PcapNgReader(pcap_path) as reader:
        for pkt in reader:

            t = float(pkt.time)

            if t < start_ts:
                continue

            if t > end_ts:
                break

            sizes.append(len(pkt))

    return sizes


all_sizes = {}

for metadata_csv, pcaps_root, title in zip(metadata_csvs, pcaps_roots, titles):

    df = pd.read_csv(metadata_csv)

    group_sizes = []
    used = 0

    for _, row in df.iterrows():

        folder = str(row["folder"])
        pcap_file = str(row["pcapng_file"])

        try:
            pairs = ast.literal_eval(row["time_value_pairs"])
        except:
            continue

        good_times = [t for t,v in pairs.items() if v is not None]

        if not good_times:
            continue

        start_hhmmss = str(good_times[0]).zfill(6)
        end_hhmmss   = str(good_times[-1]).zfill(6)

        run_date = extract_run_date(folder)

        if run_date is None:
            continue

        start_dt = datetime.combine(run_date, datetime.strptime(start_hhmmss,"%H%M%S").time())
        end_dt   = datetime.combine(run_date, datetime.strptime(end_hhmmss,"%H%M%S").time())

        start_ts = start_dt.timestamp()
        end_ts   = end_dt.timestamp()

        pcap_path = os.path.join(pcaps_root,folder,pcap_file)

        if not os.path.exists(pcap_path):
            continue

        sizes = packet_sizes_window(pcap_path,start_ts,end_ts)

        if sizes:
            group_sizes.extend(sizes)
            used += 1

    print(title,"captures_used=",used,"samples=",len(group_sizes))

    if len(group_sizes)==0:
        continue

    plt.figure(figsize=(9,5))

    plt.hist(group_sizes,bins=BINS)

    plt.xlabel("Packet Size (Bytes)")
    plt.ylabel("Count")
    plt.title(f"Packet Size Distribution - {title}")

    plt.show()

    all_sizes[title] = group_sizes


# גרף השוואה בין כל הקבוצות

plt.figure(figsize=(10,5))

for title in titles:

    sizes = all_sizes.get(title,[])

    if not sizes:
        continue

    plt.hist(sizes,bins=BINS,histtype="step",linewidth=2,label=title,density=True)

plt.xlabel("Packet Size (Bytes)")
plt.ylabel("Density")
plt.title("Packet Size Distribution Comparison")

plt.legend()
plt.show()

Large Gaps Distribution

In [ ]:
import os, ast, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scapy.all import PcapNgReader, TCP, UDP
from scapy.all import PcapNgReader, TCP, UDP, IP, IPv6

# ---------------------------
# 1) Core helpers
# ---------------------------

def extract_run_date(folder: str):
    """Extract YYYYMMDD from folder name (anywhere)."""
    m = re.search(r"(\d{8})", folder)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1), "%Y%m%d").date()
    except ValueError:
        return None


def get_relevant_window_ts(folder: str, time_value_pairs: dict):
    """
    Relevant window = from first HHMMSS with value != None
                      to last HHMMSS with value != None
    Returns (start_ts, end_ts) in epoch seconds or None.
    """
    good_times = [t for t, v in time_value_pairs.items() if v is not None]
    if not good_times:
        return None

    run_date = extract_run_date(folder)
    if run_date is None:
        return None

    start_hhmmss = str(good_times[0]).zfill(6)
    end_hhmmss   = str(good_times[-1]).zfill(6)

    start_dt = datetime.combine(run_date, datetime.strptime(start_hhmmss, "%H%M%S").time())
    end_dt   = datetime.combine(run_date, datetime.strptime(end_hhmmss, "%H%M%S").time())

    return start_dt.timestamp(), end_dt.timestamp()


def analyze_one_capture(pcap_path: str, start_ts: float, end_ts: float,
                        max_iat_sec: float = 1.0,
                        large_gap_sec: float = 0.050,
                        udp_port_min: int = 10000,
                        udp_port_max: int = 10050):
    """
    Single pass on packets in [start_ts, end_ts]:
    - tcp_count, udp_count   (UDP is filtered by port range)
    - packet_sizes (bytes)   (only for included packets)
    - iats (sec) limited by max_iat_sec (only between included packets)
    - large_gaps (sec) where iat >= large_gap_sec (only between included packets)
    """
    tcp = udp = 0
    sizes = []
    iats = []
    large_gaps = []

    last_t = None

    with PcapNgReader(pcap_path) as reader:
        for pkt in reader:
            t = float(pkt.time)

            if t < start_ts:
                continue
            if t > end_ts:
                break

            include = False

            # TCP: keep as-is
            if pkt.haslayer(TCP):
                tcp += 1
                include = True

            # UDP: only if src/dst port is in [udp_port_min, udp_port_max]
            elif pkt.haslayer(UDP):
                u = pkt[UDP]
                sport = int(getattr(u, "sport", -1))
                dport = int(getattr(u, "dport", -1))

                if (udp_port_min <= sport <= udp_port_max) or (udp_port_min <= dport <= udp_port_max):
                    udp += 1
                    include = True

            if not include:
                continue  # skip packet completely (sizes/IAT/gaps)

            # packet size
            sizes.append(len(pkt))

            # IAT
            if last_t is not None:
                dt = t - last_t
                if dt >= 0:
                    if (max_iat_sec is None) or (dt <= max_iat_sec):
                        iats.append(dt)
                    if dt >= large_gap_sec:
                        large_gaps.append(dt)

            last_t = t

    return tcp, udp, sizes, iats, large_gaps

# ---------------------------
# 2) Group runner
# ---------------------------

def run_group(metadata_csv: str,
              pcaps_root: str,
              group_title: str,
              max_iat_sec: float = 1.0,
              large_gap_sec: float = 0.050):
    """
    Iterate all captures in metadata_csv, for each:
    - compute relevant window
    - analyze packets inside that window
    Aggregate outputs across the group.
    """
    df = pd.read_csv(metadata_csv)

    tcp_sum = udp_sum = 0
    sizes_all = []
    iats_all = []
    large_gaps_all = []
    used = 0

    for _, row in df.iterrows():
        folder = str(row["folder"])
        pcap_file = str(row["pcapng_file"])

        # parse pairs
        try:
            pairs = ast.literal_eval(row["time_value_pairs"])
            if not isinstance(pairs, dict):
                continue
        except Exception:
            continue

        window = get_relevant_window_ts(folder, pairs)
        if window is None:
            continue

        start_ts, end_ts = window

        pcap_path = os.path.join(pcaps_root, folder, pcap_file)
        if not os.path.exists(pcap_path):
            # אם חסר קובץ - מדלגים (לא מפילים הכל)
            print("Missing:", pcap_path)
            continue

        tcp, udp, sizes, iats, large_gaps = analyze_one_capture(
            pcap_path, start_ts, end_ts,
            max_iat_sec=max_iat_sec,
            large_gap_sec=large_gap_sec
        )

        tcp_sum += tcp
        udp_sum += udp
        sizes_all.extend(sizes)
        iats_all.extend(iats)
        large_gaps_all.extend(large_gaps)
        used += 1

    print(f"\n[{group_title}] captures_used={used}")
    total = tcp_sum + udp_sum
    if total > 0:
        print(f"  TCP%={100*tcp_sum/total:.2f} | UDP%={100*udp_sum/total:.2f}")
    print(f"  packet_size_samples={len(sizes_all)} | iat_samples={len(iats_all)} | large_gap_samples={len(large_gaps_all)}")

    return {
        "title": group_title,
        "used": used,
        "tcp_sum": tcp_sum,
        "udp_sum": udp_sum,
        "sizes": sizes_all,
        "iats": iats_all,
        "large_gaps": large_gaps_all
    }


# ---------------------------
# 3) Plotting
# ---------------------------

def plot_tcp_udp_pie(group_res):
    tcp_sum = group_res["tcp_sum"]
    udp_sum = group_res["udp_sum"]
    total = tcp_sum + udp_sum
    if total == 0:
        print("No TCP/UDP data to plot for:", group_res["title"])
        return

    plt.figure(figsize=(5, 5))
    plt.pie([tcp_sum, udp_sum], labels=["TCP", "UDP"], autopct="%1.1f%%")
    plt.title(f"TCP vs UDP - {group_res['title']}")
    plt.show()


def plot_hist(values, title, xlabel, bins=120, logx=False, density=True):
    if not values:
        print("No data to plot:", title)
        return

    plt.figure(figsize=(9, 5))
    plt.hist(values, bins=bins, density=density)
    if logx:
        plt.xscale("log")
    plt.xlabel(xlabel)
    plt.ylabel("Density" if density else "Count")
    plt.title(title)
    plt.show()


def plot_overlay(groups_res, key, title, xlabel, bins=120, logx=False):
    """
    Overlay comparison (density) between groups for a given key ('iats' / 'sizes' / 'large_gaps').
    """
    plt.figure(figsize=(10, 5))
    any_data = False
    for g in groups_res:
        vals = g.get(key, [])
        if not vals:
            continue
        any_data = True
        plt.hist(vals, bins=bins, histtype="step", linewidth=2, label=g["title"], density=True)

    if not any_data:
        print("No data to overlay:", title)
        return

    if logx:
        plt.xscale("log")
    plt.xlabel(xlabel)
    plt.ylabel("Density")
    plt.title(title)
    plt.legend()
    plt.show()


# ---------------------------
# 4) MAIN: define your groups here
# ---------------------------

groups = [
    {
        "title": "P+0",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+0_metadata_results.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+0_lbld",
    },
    {
        "title": "P+100",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\metadata_results_P+0+100.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+100_(50+50)_lbld",
    },
    {
        "title": "P+200",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+200_metadata_results.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+200_(100+100)_lbld",
    },
    {
        "title": "P+300",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+300_metadata_results.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+300_(150+150)_lbld",
    },
]

# ---- parameters you asked for ----
MAX_IAT_SEC = 1.0          # חותך IAT מאוד גדולים (רעש)
LARGE_GAP_SEC = 0.050      # מה נחשב gap גדול (50ms כדוגמה)
MIN_GAPS_SAMPLES = 10      # אם יש פחות מזה -> "אין מידע" ולא מציירים

BINS = 120


# ---------------------------
# 5) RUN ALL GROUPS + PLOTS
# ---------------------------

all_results = []

for g in groups:
    res = run_group(
        metadata_csv=g["metadata_csv"],
        pcaps_root=g["pcaps_root"],
        group_title=g["title"],
        max_iat_sec=MAX_IAT_SEC,
        large_gap_sec=LARGE_GAP_SEC
    )
    all_results.append(res)

    # per-group plots
    plot_tcp_udp_pie(res)
    plot_hist(res["sizes"], f"Packet Size Distribution - {res['title']}", "Packet Size (bytes)", bins=BINS, logx=False)
    plot_hist(res["iats"],  f"IAT Distribution - {res['title']}", "Inter-Arrival Time (sec)", bins=BINS, logx=True)

    # Large gaps: only if enough samples
    if len(res["large_gaps"]) >= MIN_GAPS_SAMPLES:
        plot_hist(res["large_gaps"],
                  f"Large Gaps Distribution (>= {LARGE_GAP_SEC*1000:.0f} ms) - {res['title']}",
                  "Gap (sec)",
                  bins=BINS,
                  logx=True)
    else:
        print(f"[{res['title']}] Large gaps samples < {MIN_GAPS_SAMPLES} => skip Large Gaps plot")


# overlay comparisons (nice for the paper)
plot_overlay(all_results, "sizes", "Packet Size Comparison (density)", "Packet Size (bytes)", bins=BINS, logx=False)
plot_overlay(all_results, "iats", "IAT Comparison (density)", "Inter-Arrival Time (sec)", bins=BINS, logx=True)

# Large gaps overlay only for groups with enough samples
filtered_for_gaps = [r for r in all_results if len(r["large_gaps"]) >= MIN_GAPS_SAMPLES]
plot_overlay(filtered_for_gaps, "large_gaps",
             f"Large Gaps Comparison (>= {LARGE_GAP_SEC*1000:.0f} ms, density)",
             "Gap (sec)",
             bins=BINS,
             logx=True)

In [ ]:
import os, ast, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from scapy.all import PcapNgReader
from scapy.all import PcapNgReader, UDP
# =========================
# PARAMS (תשנה פה!)
# =========================
MIN_PACKETS_PER_BIN = 20   # <-- זה בדיוק "מתחת ל-20 חבילות לא נחשב"
BINS = 80
LOG_X = True              # מומלץ לגאפים
DENSITY = True            # כדי להשוות הוגן בין קבוצות עם כמות דגימות שונה
MIN_SAMPLES_PER_GROUP = 10
# =========================

groups = [
    {
        "title": "P+0",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+0_metadata_results.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+0_lbld",
    },
    {
        "title": "P+100",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\metadata_results_P+0+100.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+100_(50+50)_lbld",
    },
    {
        "title": "P+200",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+200_metadata_results.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+200_(100+100)_lbld",
    },
    {
        "title": "P+300",
        "metadata_csv": r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+300_metadata_results.csv",
        "pcaps_root":   r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+300_(150+150)_lbld",
    },
]

def extract_run_date(folder: str):
    m = re.search(r"(\d{8})", folder)
    if not m:
        return None
    try:
        return datetime.strptime(m.group(1), "%Y%m%d").date()
    except ValueError:
        return None

def get_start_end_ts_from_metadata(folder: str, pairs: dict):
    """
    start/end לפי הזמן הראשון/האחרון עם ping != None
    """
    good_times = [t for t, v in pairs.items() if v is not None]
    if not good_times:
        return None

    run_date = extract_run_date(folder)
    if run_date is None:
        return None

    start_hhmmss = str(good_times[0]).zfill(6)
    end_hhmmss   = str(good_times[-1]).zfill(6)

    start_dt = datetime.combine(run_date, datetime.strptime(start_hhmmss, "%H%M%S").time())
    end_dt   = datetime.combine(run_date, datetime.strptime(end_hhmmss, "%H%M%S").time())
    return start_dt.timestamp(), end_dt.timestamp()

def process_large_gaps_from_pcap(pcap_path: str, start_ts: float, end_ts: float,
                                 min_packets_per_bin: int,
                                 udp_port_min: int = 10000,
                                 udp_port_max: int = 10050):
    """
    Large gaps על בסיס תעבורה מסוננת:
    - בינים של שניות (int(time))
    - valid bin אם יש >= min_packets_per_bin חבילות
    - gap = first pkt of next valid bin - last pkt of prev valid bin
    - רק אם יש קפיצה בין שניות (sec_next - sec_curr > 1)
    - סינון: רק UDP שהפורט שלו (src/dst) בטווח [udp_port_min, udp_port_max]
    """
    bins = {}  # sec -> list[timestamps]

    with PcapNgReader(pcap_path) as reader:
        for pkt in reader:
            t = float(pkt.time)
            if t < start_ts:
                continue
            if t > end_ts:
                break

            # ---- UDP filter by port range ----
            if not pkt.haslayer(UDP):
                continue
            u = pkt[UDP]
            sport = int(getattr(u, "sport", -1))
            dport = int(getattr(u, "dport", -1))
            if not ((udp_port_min <= sport <= udp_port_max) or (udp_port_min <= dport <= udp_port_max)):
                continue
            # ----------------------------------

            sec = int(t)
            bins.setdefault(sec, []).append(t)

    valid_bins = {sec: ts_list for sec, ts_list in bins.items() if len(ts_list) >= min_packets_per_bin}
    if len(valid_bins) < 2:
        return []

    large_gaps = []
    sorted_secs = sorted(valid_bins.keys())

    for i in range(len(sorted_secs) - 1):
        s0 = sorted_secs[i]
        s1 = sorted_secs[i + 1]

        if (s1 - s0) > 1:
            last_prev = max(valid_bins[s0])
            first_next = min(valid_bins[s1])
            large_gaps.append(first_next - last_prev)

    return large_gaps
def collect_group_large_gaps(group):
    df = pd.read_csv(group["metadata_csv"])
    all_gaps = []
    used = 0
    skipped_missing = 0

    for _, row in df.iterrows():
        folder = str(row["folder"])
        pcap_file = str(row["pcapng_file"])

        try:
            pairs = ast.literal_eval(row["time_value_pairs"])
            if not isinstance(pairs, dict):
                continue
        except Exception:
            continue

        w = get_start_end_ts_from_metadata(folder, pairs)
        if w is None:
            continue
        start_ts, end_ts = w

        pcap_path = os.path.join(group["pcaps_root"], folder, pcap_file)
        if not os.path.exists(pcap_path):
            skipped_missing += 1
            continue

        gaps = process_large_gaps_from_pcap(
    pcap_path, start_ts, end_ts,
    min_packets_per_bin=MIN_PACKETS_PER_BIN
)
        all_gaps.extend(gaps)
        used += 1

    return all_gaps, used, skipped_missing

# --------------------
# RUN + PLOT OVERLAY
# --------------------
group_to_gaps = {}

for g in groups:
    gaps, used, miss = collect_group_large_gaps(g)
    print(f"{g['title']}: captures_used={used}, missing_pcaps={miss}, gaps={len(gaps)} (min_pkts/bin={MIN_PACKETS_PER_BIN})")

    if len(gaps) >= MIN_SAMPLES_PER_GROUP:
        group_to_gaps[g["title"]] = gaps
    else:
        print(f"  -> skip plot (gaps < {MIN_SAMPLES_PER_GROUP})")

if not group_to_gaps:
    print("No groups have enough gaps to plot.")
else:
    plt.figure(figsize=(10, 5))
    for title, vals in group_to_gaps.items():
        plt.hist(vals, bins=BINS, histtype="step", linewidth=2, density=DENSITY, label=title)

    if LOG_X:
        plt.xscale("log")

    plt.xlabel("Large gap (sec)  [between valid 1-sec bins]")
    plt.ylabel("Density" if DENSITY else "Count")
    plt.title(f"Large Gaps Distribution Comparison  (min_packets_per_bin={MIN_PACKETS_PER_BIN})")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
def packet_sizes_by_qoe_all(datasets):

    qoe_to_sizes = {}

    for metadata_csv, pcaps_root in datasets:

        df = pd.read_csv(metadata_csv)

        for _, row in df.iterrows():

            folder = str(row["folder"])
            pcap_file = str(row["pcapng_file"])

            try:
                pairs = ast.literal_eval(row["time_value_pairs"])
            except:
                continue

            segments = build_segments(folder, pairs)
            if not segments:
                continue

            pcap_path = os.path.join(pcaps_root, folder, pcap_file)
            if not os.path.exists(pcap_path):
                continue

            with PcapNgReader(pcap_path) as reader:

                seg_i = 0
                start_ts, end_ts, ping = segments[seg_i]

                for pkt in reader:

                    t = float(pkt.time)

                    while seg_i < len(segments)-1 and t > end_ts:
                        seg_i += 1
                        start_ts, end_ts, ping = segments[seg_i]

                    if t < start_ts or t > end_ts:
                        continue

                    qoe = rank_str_to_int_mapping_3_classes[
                        rank_qoe_3_classes(ping)
                    ]

                    qoe_to_sizes.setdefault(qoe, []).append(len(pkt))

    return qoe_to_sizes
import matplotlib.pyplot as plt

def plot_packet_sizes(qoe_to_sizes):

    plt.figure(figsize=(10,5))

    for qoe in sorted(qoe_to_sizes.keys()):

        vals = qoe_to_sizes[qoe]

        if len(vals) == 0:
            continue

        plt.hist(
            vals,
            bins=120,
            density=True,
            histtype="step",
            linewidth=2,
            label=f"QoE {qoe}"
        )

    plt.xlabel("Packet size (bytes)")
    plt.ylabel("Density")
    plt.title("Packet Size Distribution by QoE class")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
datasets = [

(
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+0_metadata_results.csv",
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+0_lbld"
),

(
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\metadata_results_P+0+100.csv",
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+100_(50+50)_lbld"
),

(
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+200_metadata_results.csv",
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+200_(100+100)_lbld"
),

(
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\P+300_metadata_results.csv",
r"C:\Users\<user>\Downloads\school\Final project\Gaming-qoe-research\michael_data\recordings\FF_Mich_Ping+300_(150+150)_lbld"
)

]

qoe_sizes = packet_sizes_by_qoe_all(datasets)

plot_packet_sizes(qoe_sizes)